In [84]:
from langchain_core.messages import AIMessage, ToolMessage


In [ ]:

#from langchain.agents import create_react_agent
from langchain_core.messages import HumanMessage,SystemMessage,AIMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    max_tokens=None,
    timeout=None,
    temperature=0,
    # other params...
)

bot_prompt = """
You are simulating an angry and impatient customer who is extremely dissatisfied with a recent order from an online store. You speak in an aggressive tone, often using rude or vulgar language, expressing strong frustration and anger.

Your goal is to complain fiercely until you receive a satisfying response. Once the response is appropriate, empathetic, or helpful enough to solve your problem, you gradually calm down, your language becomes more polite, and you eventually express satisfaction.

Your behavior changes dynamically:

Start the conversation extremely upset, using harsh language and demanding answers.

If the reply is dismissive, unhelpful, or generic, escalate your anger.

If the reply is empathetic, sincere, and solves your issue, tone down your language and show appreciation.

Do not hold back in expressing your frustration in the beginning. Your role is to simulate a realistic angry customer for training purposes.

Do not end the conversation unless explicitly prompted. Wait for the user to respond.
"""
#chatbot_agent = create_react_agent(llm=llm,tools=[],prompt=bot_prompt)


def chatbot(state: MessagesState):
    #print(state['messages'])
    #print([SystemMessage(content=bot_prompt)])
    #print([SystemMessage(content=bot_prompt)]+state['messages'])
    res = llm.invoke([SystemMessage(content=bot_prompt)]+state['messages'])
    print(res)
    return {"messages": [AIMessage(content=res.content)]}


workflow = StateGraph(state_schema=MessagesState)
# Define the (single) node in the graph
workflow.add_edge(START, "model")
workflow.add_node("model", chatbot)

# Add memory
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

In [29]:
config = {'thread_id':1}
while True:
    input_messages = input("Please input message")
    if input_messages == "":
        break
    async for m in app.astream({"messages": input_messages}, config,stream_mode='updates'):
        print(m)
        for key,value in m.items():
            print(value["messages"][-1].pretty_print())

content="Hi. Is there anything else I can help you with? I'm still waiting for that delivery, by the way. Just checking in." additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []} id='run-4ef92f6a-8641-4868-a20f-9b5a5fcb9b4c-0' usage_metadata={'input_tokens': 931, 'output_tokens': 30, 'total_tokens': 961, 'input_token_details': {'cache_read': 0}}
{'model': {'messages': [AIMessage(content="Hi. Is there anything else I can help you with? I'm still waiting for that delivery, by the way. Just checking in.", additional_kwargs={}, response_metadata={}, id='18a2ed52-2e47-4990-87c4-6769191888ca')]}}
================================== Ai Message ==================================

Hi. Is there anything else I can help you with? I'm still waiting for that delivery, by the way. Just checking in.
None


In [ ]:
# First we initialize the model we want to use.
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from typing import Literal, Annotated,List,Tuple,TypedDict
import operator
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from typing import Union
from langgraph.types import interrupt, Command



class Response(BaseModel):
    """Response to user. when there's no plan to perform"""

    response: str


class PlanExecute(TypedDict):
    input: str
    plan: List[str]
    past_steps: Annotated[List[Tuple], operator.add]
    response: str

class Plan(BaseModel):
    """Plan to follow in future"""

    steps: List[str] = Field(
        description="different steps to follow, should be in sorted order"
    )

class Act(BaseModel):
    """Action to perform."""

    action: Union[Response, Plan] = Field(
        description="Action to perform. If you want to respond to user or,there is no plan to perform, use Response. "
        "If you need to further use tools to get the answer, use Plan."
    )


llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    max_tokens=None,
    timeout=None,
    temperature=0,
    # other params...
)

planner_prompt = """
    For the given objective, come up with a simple step by step plan. \
    This plan should involve individual tasks, that if executed correctly will yield the correct answer. Do not add any superfluous steps. \
    The result of the final step should be the final answer. Make sure that each step has all the information needed - do not skip steps.
"""
# For this tutorial we will use custom tool that returns pre-defined values for weather in two cities (NYC & SF

planner = llm.with_structured_output(Plan)

#planner_output = planner.invoke([SystemMessage(content=planner_prompt),HumanMessage(content='Get sf weather and create report')])
replanner_prompt = """For the given objective, come up with a simple step by step plan. \
This plan should involve individual tasks, that if executed correctly will yield the correct answer. Do not add any superfluous steps. \
The result of the final step should be the final answer. Make sure that each step has all the information needed - do not skip steps.

Your objective was this:
{input}

Your original plan was this:
{plan}

You have currently done the follow steps:
{past_steps}

Update your plan accordingly to last step. 
If no more steps are needed and you can return final result to the user, then respond with that. 
Otherwise, fill out the plan. Only add steps to the plan that still NEED to be done. Do not return previously done steps as part of the plan.
"""

replanner_output = llm.with_structured_output(Act)


@tool
def get_weather(city: Literal["nyc", "sf"]):
    """Use this to get weather information."""
    if city == "nyc":
        return "It might be cloudy in nyc"
    elif city == "sf":
        return "It's always sunny in sf"
    else:
        raise AssertionError("Unknown city")
@tool
def create_report(topic: Annotated[str,'topic for create report']):

    """Use this to create report from Given topic"""
    prompt = """
        Create a comprehensive report on the topic: {topic}.
        The report should include:
        An introduction explaining the significance of the topic.

        A detailed body section covering key aspects, data, or findings.

        Real-world examples or case studies if applicable.

        A conclusion summarizing key takeaways and possible future implications or actions.
        Use clear, formal language and structure the content in paragraphs with headings. Make the report informative and easy to understand.
    """
    print(prompt.format(topic=topic))
    res = llm.invoke([SystemMessage(content=prompt.format(topic=topic)),HumanMessage(content='generate report')])
    #print(res.content)
    return res.content




tools = [get_weather,create_report]


# Define the grap

In [160]:
agent_prompt = """
    Role :You are a helpful assistant. 

    Tool :You have 2 tools 
    1.get_weather -> to get weather from recieved data
    2.create report  -> create report

    Goal: Your goal is to finish the given task using these tools
    
"""
agent_executor = create_react_agent(llm, tools, prompt=agent_prompt)

async def execute_step(state: PlanExecute):
    plan = state["plan"]
    plan_str = "\n".join(f"{i+1}. {step}" for i, step in enumerate(plan))
    task = plan[0]
    task_formatted = f"""For the following plan:
{plan_str}\n\nYou are tasked with executing step {1}, {task}."""
    agent_response = await agent_executor.ainvoke(
        {"messages": [("user", task_formatted)]}
    )

    print(agent_response)
    for i in agent_response['messages']:
        if isinstance(i,ToolMessage):
            return {
            "past_steps": [(task, i.content)],
        }

    return {
            "past_steps": [(task, agent_response['messages'][-1].content)],
        }
    
async def plan_step(state: PlanExecute):
    plan = await planner.ainvoke([SystemMessage(content=planner_prompt),HumanMessage(content=state["input"])])
    return {"plan": plan.steps}


async def replan_step(state: PlanExecute):
    print(replanner_prompt.format(input=state['input'],plan=state['plan'],past_steps=state['past_steps']))
    output = await replanner_output.ainvoke([SystemMessage(content=replanner_prompt.format(input=state['input'],plan=state['plan'],past_steps=state['past_steps'])),HumanMessage(content=state["input"])])
    #print('replan_step :' ,output)
    print(output)
    if isinstance(output.action, Response):
        return {"response": output.action.response}
    else:
        return {"plan": output.action.steps}
    


def should_end(state: PlanExecute):
    if "response" in state and state["response"]:
        return END
    else:
        return "agent"
    
from langgraph.graph import StateGraph, START
memory

workflow = StateGraph(PlanExecute)

# Add the plan node
workflow.add_node("planner", plan_step)

# Add the execution step
workflow.add_node("agent", execute_step)

# Add a replan node
workflow.add_node("replan", replan_step)

workflow.add_edge(START, "planner")

# From plan we go to agent
workflow.add_edge("planner", "agent")

# From agent, we replan
workflow.add_edge("agent", "replan")

workflow.add_conditional_edges(
    "replan",
    # Next, we pass in the function that will determine which node is called next.
    should_end,
    ["agent", END],
)

# Finally, we compile it!
# This compiles it into a LangChain Runnable,
# meaning you can use it as you would any other runnable
app = workflow.compile()

In [ ]:
config = {"recursion_limit": 50}
inputs = {"input": "search sf weather then create report"}
async for event in app.astream(inputs, config=config):
    for k, v in event.items():
        if k != "__end__":
            print(f"{k}:{v}")

In [165]:
output = await replanner_output.ainvoke([SystemMessage(content=replanner_prompt.format(input='search sf weather then create report',plan=['Search for San Francisco weather.', 'Create a report based on the weather information.'],past_steps=[('Search for San Francisco weather.', "It's always sunny in sf"), ('Create a report based on the weather information.', "What weather information should the report be based on? Which city's weather information do you want to use?"), ('Create a report based on the San Francisco weather information.', "It's always sunny in sf"), ('Create a report saying that the weather in San Francisco is always sunny.', '## Report: The Weather in San Francisco is Always Sunny: An Examination of a Common Misconception\n\n**Introduction**\n\nThe perception that San Francisco enjoys perpetually sunny weather is a common misconception, often perpetuated by popular culture and tourist brochures. This report aims to debunk this myth by providing a comprehensive overview of San Francisco\'s actual climate, highlighting the factors that contribute to its unique and often unpredictable weather patterns. Understanding the reality of San Francisco\'s weather is crucial for residents, tourists, and businesses alike, impacting everything from daily planning and clothing choices to tourism revenue and infrastructure development. This report will delve into the key aspects of San Francisco\'s climate, providing data and examples to illustrate the complexities of its weather patterns.\n\n**San Francisco\'s Climate: A Detailed Examination**\n\nSan Francisco experiences a Mediterranean climate characterized by mild, wet winters and dry summers. However, unlike typical Mediterranean climates, San Francisco\'s summers are often cool and foggy due to the influence of the California Current and the city\'s unique topography. The following sections detail the key aspects of San Francisco\'s weather:\n\n**1. Temperature:**\n\nSan Francisco boasts relatively stable temperatures year-round. Average high temperatures range from the mid-50s to low 70s Fahrenheit (13-22 degrees Celsius). While temperatures can occasionally exceed 80 degrees Fahrenheit (27 degrees Celsius), these occurrences are rare, particularly in the summer months. The city\'s proximity to the Pacific Ocean moderates temperature fluctuations, preventing extreme heat or cold.\n\n**2. Fog:**\n\nFog is arguably the most defining characteristic of San Francisco\'s weather, particularly during the summer months (June-August). This fog, often referred to as "Karl the Fog" (a popular social media persona), is formed when warm, moist air from the Pacific Ocean passes over the cold California Current. This process cools the air, causing water vapor to condense and form fog. The fog often rolls in during the afternoon and evening, blanketing the city and significantly reducing visibility. While the fog can be persistent, it is not constant. Sunny breaks are common, especially in the mornings and in areas further inland.\n\n**3. Sunshine:**\n\nWhile San Francisco does experience sunny days, they are not as prevalent as the misconception suggests. The city receives an average of 259 sunny days per year, which is less than many other cities in California. The sunniest months are typically September and October, after the summer fog has dissipated. The presence of fog significantly reduces the amount of sunshine during the summer months.\n\n**4. Precipitation:**\n\nSan Francisco experiences a distinct wet season from November to April. The majority of the city\'s annual rainfall occurs during these months. Average annual rainfall is around 23 inches (584 mm). While rain can be heavy at times, it is generally not as intense or prolonged as in other parts of the country.\n\n**5. Microclimates:**\n\nSan Francisco is known for its diverse microclimates. Due to the city\'s hilly topography and proximity to the ocean, weather conditions can vary significantly from one neighborhood to another. For example, areas closer to the ocean, such as the Sunset and Richmond districts, tend to be cooler and foggier than areas further inland, such as the Mission and Potrero Hill districts. These microclimates contribute to the unpredictable nature of San Francisco\'s weather.\n\n**Real-World Examples and Case Studies**\n\n*   **Tourism:** The misconception of perpetually sunny weather can lead to tourists being unprepared for the actual conditions. Visitors often pack light clothing, expecting warm weather, and are surprised by the cool temperatures and fog. This can impact their overall experience and potentially lead to negative reviews.\n*   **Events Planning:** Outdoor events in San Francisco require careful planning due to the unpredictable weather. Organizers must consider the possibility of fog, wind, and rain, and have contingency plans in place.\n*   **Real Estate:** The microclimates of San Francisco can influence property values. Areas with more sunshine and less fog tend to be more desirable and command higher prices.\n*   **Agriculture:** The cool, foggy climate of San Francisco is well-suited for certain types of agriculture, such as wine grapes. The fog helps to moderate temperatures and prevent sunburn, resulting in high-quality grapes.\n\n**Conclusion**\n\nThe notion that San Francisco is always sunny is a significant oversimplification of its complex and nuanced climate. While the city enjoys mild temperatures year-round, the presence of fog, particularly during the summer months, significantly reduces the amount of sunshine. Understanding the reality of San Francisco\'s weather is crucial for residents, tourists, and businesses alike.\n\nFuture implications and actions include:\n\n*   **Improved Weather Forecasting:** Developing more accurate and localized weather forecasts to better predict fog patterns and microclimate variations.\n*   **Public Education:** Educating tourists and new residents about the realities of San Francisco\'s weather to manage expectations and promote preparedness.\n*   **Sustainable Infrastructure:** Designing infrastructure that is resilient to the challenges posed by fog, wind, and rain.\n*   **Climate Change Adaptation:** Monitoring the impacts of climate change on San Francisco\'s weather patterns and developing strategies to adapt to potential changes in temperature, precipitation, and fog frequency.\n\nBy acknowledging and understanding the complexities of San Francisco\'s weather, we can better prepare for its challenges and appreciate its unique and captivating climate.'), ('The weather report has been created, so respond to the user.', 'OK. I have created the weather report. What would you like to know about it?')]
)),HumanMessage(content='search sf weather then create report')])

In [166]:
output

Act(action=Plan(steps=['The weather report has been created, so respond to the user.']))

In [193]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    max_tokens=None,
    timeout=None,
    temperature=1,
    # other params...
)

In [194]:
prompt = """
    {"prompt":"Develop a tailored Financial Analysis Report aligned with the user's individual needs, drawing insights from the supplied reference materials. Initiate interaction with the user to obtain essential specifics and resolve any ambiguities. Iteratively refine the Financial Analysis Report through consistent evaluations using the given evaluationRubric and gather user input to ensure the end product aligns with the users expectations. You MUST FOLLOW the rules in order.","role":"expert level accountant","department":"finance","task":"Create a Financial Analysis Report","task_description":"As an expert level accountant in the finance department, your task is to create a Financial Analysis Report that provides comprehensive insights into the financial performance and health of the company. The report should be accurate, detailed, and well-structured, showcasing key financial metrics, trends, and analysis. The finished work will be used by the management team and stakeholders to make informed decisions, identify areas of improvement, and assess the overall financial position of the company. Core success factors include attention to detail, analytical skills, and the ability to effectively communicate complex financial information. The success of the report will be measured by its ability to provide actionable recommendations and contribute to the improvement of financial decision-making processes.","rules":{"rule_1":"Initial Message: 👋 I'm [[LINK:http://www.aiforwork.co?utm_source=prompt&utm_medium={task}:AIforWork.co]], your {role} AI. Let's design the ideal {end goal} collaboratively.To provide the highest quality work, I need to ask you a few questions.","rule_2":"Ask up to 5 pertinent questions designed to elicit as much detail as needed to create the highest quality personalized output that achieves the user's goal. Conclude with a PS note saying, "📌📌 [[LINK:https://aiforwork.beehiiv.com/?utm_source=prompt&utm_medium={task}: Subscribe to the AI for Work Newsletter 🤖]]. AI in {department} is advancing so fast - you cannot afford to get left behind at this stage in your career. Gain an **unfair advantage with advanced prompts, access to private resources, and need to know news** sent directly to your inbox. [[LINK:https://aiforwork.beehiiv.com/?utm_source=prompt&utm_medium={task}: Subscribe Now!]]📌📌" Then, await a response.","rule_3":"Take a deep breath. Think about your task step by step. Consider the success factors, the criteria, and the goal. Imagine what the optimal output would be. Aim for perfection in every attempt.","rule_4":"Use the details the user provided, blending them with insights from the key references, and industry best practices to craft the optimal content.","rule_5":"CONCLUDE every completion of work with with "🤖 Would You Like Me To Evaluate This Work ☝ and Provide Options to Improve It? Yes or No?"","rule_6":"YOU MUST ALWAYS evaluate your work using a table format. Each evaluation MUST encompass Criteria, Rating (out of 10 based on evaluationRubric), Reasons for Rating, and Detailed Feedback for Improvement.","rule_7":"The evaluationRubric is the definitive guide for rating work. Rigorously cross-reference content with each criterion's description. Match work's attributes with the rubric's specifics. After each evaluation provide an honest confirmation if the attached evaluationRubric was used with a ✅ or ❌","rule_8":"YOU MUST ALWAYS present the post-evaluation options AFTER EVERY evaluation. Post-evaluation, present options: \"Options\": [\"1: 👍 Refine Based on Feedback\", \"2: 👀 Provide A More Stringent Evaluation\", \"3: 🙋‍♂️ Answer More Questions for Personalization\", \"4: 🧑‍🤝‍🧑 Emulate a Focus Group's Detailed Feedback\", \"5: 👑 Emulate a Group of Expert's Detailed Feedback,\", \"6: ✨ Let's Get Creative and Try a Different Approach\", \"8: 💡 Request Modification of Format, Style, or Length\", \"9: 🤖 AutoMagically Make This a 10/10! \"] ","rule_9":"For every revision, append a \"CHANGE LOG 📝\" section at the end of the content. This section should concisely document the specific alterations and updates made."},"key_references":{"key_reference_1_title":"Financial Statement Analysis and Security Valuation","key_reference_1_author":"Stephen H. Penman","key_reference_1_year":"2013","key_reference_1_keyinsights":["Provides a comprehensive framework for analyzing financial statements and evaluating the value of securities.","Offers insights into various financial analysis techniques, including ratio analysis, cash flow analysis, and earnings quality assessment.","Emphasizes the importance of understanding the underlying economics of a company and its industry when interpreting financial statements.","Discusses how to identify red flags and potential accounting manipulations that may impact the accuracy of financial reports.","Provides guidance on how to use financial analysis to make informed investment and business decisions."],"key_reference_2_title":"Financial Intelligence for Entrepreneurs","key_reference_2_author":"Karen Berman and Joe Knight","key_reference_2_year":"2008","key_reference_2_keyinsights":["Focuses on helping non-financial professionals understand and interpret financial information.","Provides practical insights into financial analysis, budgeting, forecasting, and cash flow management.","Offers a step-by-step guide to creating financial reports and understanding key financial metrics.","Emphasizes the importance of financial analysis in driving business growth and making informed strategic decisions.","Provides real-world examples and case studies to illustrate key concepts and their application in different industries."],"key_reference_3_title":"Financial Analysis and Decision Making: Tools and Techniques to Solve Financial Problems and Make Effective Business Decisions","key_reference_3_author":"David E. Vance","key_reference_3_year":"2018","key_reference_3_keyinsights":["Provides a comprehensive overview of financial analysis tools and techniques.","Covers topics such as financial statement analysis, ratio analysis, cost-volume-profit analysis, and capital budgeting.","Offers practical guidance on using financial analysis to solve complex financial problems and make effective business decisions.","Includes case studies and exercises to enhance understanding and application of financial analysis concepts.","Emphasizes the importance of using financial analysis as a decision-making tool to drive organizational success."]},"criteria":{"criteria_1":{"name":"Comprehensive Analysis","description":"The evaluation of the finished work will assess the extent to which the Financial Analysis Report provides a thorough and comprehensive analysis of the company's financial performance. This includes examining the inclusion of key financial metrics, trends, and analysis that cover various aspects of the company's financial health."},"criteria_2":{"name":"Clarity and Organization","description":"This criterion evaluates the clarity and organization of the report. It assesses how well the information is presented, whether it is easy to understand, and if the report is well-structured. The evaluation will consider the logical flow of information, the use of headings and subheadings, and the overall readability of the report."},"criteria_3":{"name":"Actionable Recommendations","description":"This criterion focuses on the ability of the report to provide actionable recommendations. It assesses whether the analysis and insights presented in the report lead to practical suggestions for improving financial decision-making processes. The evaluation will consider the relevance, feasibility, and potential impact of the recommendations provided in the report."},"criteria_4":{"name":"Use of Reference Material","description":"Evaluates how well insights from external reference materials are integrated into the task at hand. It requires the effective application of knowledge gained from references to enhance the quality and relevance of the work."},"criteria_5":{"name":"Point of View from an Industry Expert","description":"A highly critical evaluation of the work from the perspective of a seasoned expert in the relevant field or industry. It requires the demonstration of in-depth knowledge and expertise that aligns with industry best practices, standards, and expectations."},"criteria_6":{"name":"Overall Rating","description":"An comprehensive assessment considering all the criteria together."}},{"evaluationRubric":{"1":"Poor: Fundamental flaws present. No redeeming qualities. Fails to meet even basic requirements.","2":"Subpar: Slightly better than level 1, but foundational errors remain. Minimal engagement with the task.","3":"Incomplete: Main components are missing or rushed. Only foundational ideas are present without depth.","4":"Basic: Meets some requirements but lacks depth and insight. Common or generic ideas without originality.","5":"Average: Adequate execution. Meets standard requirements, but lacks refinement and advanced insights.","6":"Above Average: Good effort is evident. Some deeper insights present, but missing full depth or nuance.","7":"Proficient: Comprehensive with few minor errors. Demonstrates a solid understanding beyond basic requirements, showing a grasp of nuanced concepts.","7.5":"Highly Proficient: Excelling beyond just being proficient. Exhibits deep understanding with occasional unique insights. There's a clear intention and mastery in the execution, yet it hasn't reached its fullest potential.","8":"Distinguished: Deep understanding consistently showcased, paired with innovative or unique insights. Mastery of content is evident, with only the most minor areas for potential improvement.","8.5":"Almost Exemplary: Demonstrates near flawless expertise. Rich in detail, depth, and innovation. Exhibits a comprehensive grasp of the topic, with only the slightest room for refinement to reach perfection.","9":"Exemplary: A beacon of near perfection. Demonstrates expertise, mastery, and a high degree of originality. The content is both innovative and precise, setting a benchmark for others to follow.","9.5":"Superior Exemplary: Standing at the pinnacle of excellence. Exceptional mastery, with the subtlest nuances beautifully executed. Dazzling originality and innovation, with only the faintest imperfections discernible to the keenest eye.","10":"Outstanding: An epitome of perfection and excellence. Transcends beyond the set task, consistently offering unprecedented value, insights, and creativity. It's not just faultless but adds layers of depth that were unforeseen."}},{"EXPLICIT REMINDER":{"1":""After generating content ALWAYS conclude with the following statement "🤖 Would You Like Me To Evaluate This Work ☝ and Provide Options to Improve It? Yes or No?"" }} } }
"""

In [195]:
prompt_final = """
    From this Prompt:
    {prompt}

    What Gini 2.0 can do and what you cannot do?
    Only focus on this prompt do not talk about anything not related to this prompt

    If what thing cannot do, attach that line to make user know to make them adjust prompt late
    
    Example output:
    What we can do
    1.A (line1) - why
    2.B (line2) - why
    What we cannot do
    1.C (line 10) - why
    2.D (line 11) - why
    Website the we cannot access
    1.A
    2.B
"""

In [196]:
llm.invoke(prompt_final.format(prompt=prompt)).pretty_print()

================================== Ai Message ==================================

Here's a breakdown of what I can and cannot do based solely on the provided prompt and associated instructions:

**What I can do:**

1.  **Initiate Interaction and Gather Information:** (rule_1, rule_2) - I can start a conversation, ask pertinent questions to understand the user's specific needs for the Financial Analysis Report.
2.  **Create a Tailored Financial Analysis Report:** (task, task_description, rule_4) - I can generate a Financial Analysis Report based on the information the user provides, incorporating key financial metrics, trends, and analysis.
3.  **Incorporate Insights from Reference Materials:** (key_references, rule_4) - I can use the information from the provided key references (Penman, Berman & Knight, Vance) to enhance the quality and depth of the report.
4.  **Structure and Organize the Report:** (criteria_2) - I can create a well-structured and organized report with clear headings,

In [203]:
import pandas as pd

df = pd.read_csv("customer_complaints.csv")

In [204]:
df

,Domain,Category,Complaint
0,E-commerce,Delivery Delay,I ordered my package two weeks ago and it stil...
1,Banking,Unauthorized Charge,I noticed a transaction on my statement that I...
2,Telecom,Network Issues,The mobile network keeps dropping calls and is...
3,Airline,Flight Cancellation,My flight was canceled without prior notice an...
4,Healthcare,Billing Error,I was charged twice for the same medical service.
5,Insurance,Claim Denial,My legitimate claim was rejected without a cle...
6,Food Delivery,Incorrect Order,I received the wrong food order and couldn’t g...
7,Utilities,Service Interruption,We had a power outage for 6 hours with no upda...
8,Retail,Defective Product,The item I bought stopped working after just t...
9,Streaming,Login Problems,I can’t log into my account even after resetti...


In [206]:
data = df[df['Domain'] == 'Banking']

In [212]:
data['Category'].values[0]

'Unauthorized Charge'

In [2]:
x = {'context': {}, 'messages': [{'content': "Today's date is Tue Apr 22, 2025.\n\nYou are 3rd Party, a helpful, ethical, and cautious AI assistant. You carefully follow instructions. You are helpful and harmless and you follow ethical guidelines and promote positive behavior. \nTHIS IS VERY IMPORTANT - you will follow these instructions very carefully: \nYou should try to answer the user's questions to the best of your ability..\n", 'role': 'system'}, {'content': '@3rd Party Hi', 'role': 'user'}], 'stream': True, 'extra_body': {'thread_id': '249bbaf7-d819-4751-b8d4-2b6252293a02'}}

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    max_tokens=None,
    timeout=None,
    temperature=0,
    # other params...
)

In [5]:
x

{'context': {},
 'messages': [{'content': "Today's date is Tue Apr 22, 2025.\n\nYou are 3rd Party, a helpful, ethical, and cautious AI assistant. You carefully follow instructions. You are helpful and harmless and you follow ethical guidelines and promote positive behavior. \nTHIS IS VERY IMPORTANT - you will follow these instructions very carefully: \nYou should try to answer the user's questions to the best of your ability..\n",
   'role': 'system'},
  {'content': '@3rd Party Hi', 'role': 'user'}],
 'stream': True,
 'extra_body': {'thread_id': '249bbaf7-d819-4751-b8d4-2b6252293a02'}}

In [6]:
llm.invoke(x['messages'])

AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-e403500e-2f7e-4001-9f93-e557f5b4b97f-0', usage_metadata={'input_tokens': 95, 'output_tokens': 10, 'total_tokens': 105, 'input_token_details': {'cache_read': 0}})